Importing requried libaries

In [0]:
import sys
sys.path.append('/Workspace/Users/saik84328@gmail.com')

from pyspark.sql import functions as F
from delta.tables import DeltaTable
from SetUp.Config import bronze_schema, silver_schema, gold_schema
from pyspark.sql.types import *
from datetime import datetime
import uuid


In [0]:
start_time = datetime.now()

In [0]:
%run /Workspace/Users/saik84328@gmail.com/DataBricksLearning/AuditData

In [0]:
%run /Workspace/Users/saik84328@gmail.com/DataBricksLearning/ValidationFramework

Reading paryetransaction data

In [0]:
df_prayerTran = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load([
        "/Volumes/helathcare_bronze/default/helathcare/payer_transitions.csv",
    ])

display(df_prayerTran)

Reading Prayer Details


In [0]:
df_prayer = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load([
        "/Volumes/helathcare_bronze/default/helathcare/payers.csv",
    ])

display(df_prayer)

Combined Prayer and Prayer transaction details


In [0]:


Joined_df = F.broadcast(
    df_prayer.alias("p")
).join(
    
    df_prayerTran.alias("c"),
    
    F.col("p.Id") == F.col("c.PAYER"),
    
    "inner"
)

display(Joined_df)

Selecting requried Columns

In [0]:
df_bronze=Joined_df.select(df_prayer.Id.alias("Id"),df_prayer.NAME.alias("Insurance_provider"),df_prayer.OWNERSHIP.alias("Organization_name"),df_prayer.ADDRESS.alias("Insurance_address"),df_prayer.CITY.alias("Insurance_city"),df_prayer.STATE_HEADQUARTERED.alias("Insurance_state"),df_prayer.PHONE.alias("Inurance_phone"),df_prayer.AMOUNT_COVERED.alias("Amount_covered"),df_prayer.AMOUNT_UNCOVERED.alias("Amount_uncovered"),df_prayer.REVENUE.alias("Revenue"),df_prayer.COVERED_ENCOUNTERS.alias("Covered_encounters"),df_prayer.UNCOVERED_ENCOUNTERS.alias("Uncovered_encounters"),df_prayer.COVERED_MEDICATIONS.alias("Covered_medications"),df_prayer.UNCOVERED_MEDICATIONS.alias("Uncovered_medications"),df_prayer.COVERED_PROCEDURES.alias("Covered_procedures"),df_prayer.UNCOVERED_PROCEDURES.alias("Uncovered_procedures"),df_prayer.COVERED_IMMUNIZATIONS.alias("Covered_immunizations"),df_prayer.UNCOVERED_IMMUNIZATIONS.alias("Uncovered_immunizations"),df_prayer.UNIQUE_CUSTOMERS.alias("Unique_customers"),df_prayer.QOLS_AVG.alias("Qols_avg"),df_prayer.MEMBER_MONTHS.alias("Member_months"),df_prayerTran.PATIENT.alias("Patient_id"),df_prayerTran.MEMBERID.alias("Member_id"))
                                                                                                            
display(df_bronze)

Creating Bronze Table

In [0]:

# Drop table if exists to avoid metadata mismatch
#spark.sql(f"DROP TABLE IF EXISTS helathcare_bronze.{bronze_schema}.PatientData")

df_bronze.write.format("delta").option("delta.enableChangeDataFeed","true").mode("overwrite").saveAsTable(f"helathcare_bronze.{bronze_schema}.InuranceDetails")

In [0]:
%sql

SELECT count(*) FROM `helathcare_bronze`.`helathcare_bronze`.`InuranceDetails`

Silver Validation


Checking Nulls

In [0]:
#Checking for null values in the bronze table
silver_df = spark.table("helathcare_bronze.helathcare_bronze.InuranceDetails")
#conclusion


In [0]:
validation_result = run_validations(
    silver_df,
    ["Patient_id"]
)

print(f"Duplicate Count : {validation_result['duplicates']}")

print(
    f"Primary Key Status : "
    f"{validation_result['primary_key']['status']}"
)

print("\nColumns Having Null Values:")

display(validation_result["nulls"])


#conclusion
# --in DRIVERS have null values,so fill with seuence so i taken min and max for that increase max valu by 1 --max S99999871,min-S99911728
# ---PREFIX have null based on gender column need to fill mr or mis in perfix
#---FIPS have nulls values , need to fill with county name having filps code
#

Handling Duplicates

In [0]:
# #Drop Duplicates using hash key
print(f"Before Count: {silver_df.count()}")
silver_df=silver_df.dropDuplicates(["Patient_id"])
print(f"After Count: {silver_df.count()}")


Standlize the data

Standardized text by converting the first character of each record to uppercase.

In [0]:
silver_df = standardize_string_columns(silver_df)

In [0]:
#create tempview
silver_df.createOrReplaceTempView(
    "source_prayer"
)


In [0]:
# silver_df.write.format("delta") \
#     .option("delta.enableChangeDataFeed", "true") \
#     .option("mergeSchema", "true") \
#     .mode("append") \
#     .saveAsTable(f"helathcare_silver.{silver_schema}.SL_insurancedeatils")

In [0]:
%sql
select * from  helathcare_silver.helathcare_silver.SL_insurancedeatils

In [0]:
validation_result = run_validations(
    silver_df,
    ["Patient_id"]
)

print(f"Duplicate Count : {validation_result['duplicates']}")

print(
    f"Primary Key Status : "
    f"{validation_result['primary_key']['status']}"
)

print("\nColumns Having Null Values:")

display(validation_result["nulls"])


#conclusion
# --in DRIVERS have null values,so fill with seuence so i taken min and max for that increase max valu by 1 --max S99999871,min-S99911728
# ---PREFIX have null based on gender column need to fill mr or mis in perfix
#---FIPS have nulls values , need to fill with county name having filps code
#

In [0]:
Insurance_count=silver_df.count()
print(f"Total Count: {Insurance_count}")

In [0]:
%sql
MERGE INTO helathcare_silver.helathcare_silver.SL_insurancedeatils tgt

USING source_prayer src

ON tgt.Patient_id= src.Patient_id

WHEN MATCHED THEN
UPDATE SET *

WHEN NOT MATCHED THEN
INSERT *

In [0]:
end_time = datetime.now()

duration_seconds = int(
    (end_time - start_time).total_seconds()
)

print(duration_seconds)

In [0]:
from pyspark.sql.types import LongType
from datetime import datetime

# Get Workflow Run ID
try:
    run_id = dbutils.jobs.taskContext().taskRunId()
except:
    run_id = f"MANUAL_{datetime.now().strftime('%Y%m%d%H%M%S')}"

target_table = "helathcare_silver.helathcare_silver.SL_insurancedeatils"

# Get metadata
notebook_name, table_name, layer = get_audit_metadata(target_table)

status = "SUCCESS"
error_message = None
record_count = 0

In [0]:
# Duplicate Check

duplicate_check_status = (
    "PASS"
    if validation_result["duplicates"] == 0
    else "FAIL"
)

# Primary Key Check

primary_key_status = (
    validation_result["primary_key"]["status"]
)

# Null Check

null_count = (
    validation_result["nulls"]
    .agg(F.sum("null_count"))
    .collect()[0][0]
)

null_check_status = (
    "PASS"
    if null_count == 0
    else "FAIL"
)

# Standardization

standardization_status = "PASS"

In [0]:
end_time = datetime.now()

write_audit(
    target_table=target_table,
    run_id=run_id,
    record_count=Insurance_count,
    start_time=start_time,
    end_time=end_time,
    status=status,
    duplicate_check_status=duplicate_check_status,
    primary_key_status=primary_key_status,
    null_check_status=null_check_status,
    standardization_status=standardization_status,
    error_message=error_message
)